In [5]:
import psycopg2
import pandas as pd
from datetime import datetime
from psycopg2.extras import execute_values

new_loans = []
existing_loans = []

conn = psycopg2.connect(
    dbname='postgres',
    host='localhost',
    user='postgres',
    password=input()
)

cur = conn.cursor()

query = """
SELECT disclosure_sequence_number,
issuer_id,
state_code,
msa,
original_principal_balance,
loan_interest_rate,
ltv,
credit_score,
loan_origination_date,
down_payment_assistance,
first_time_home_buyer,
seller_issuer_id,agency
FROM mbs_loans"""

temp_df = pd.read_sql(query, conn)

print(f"Loaded {len(temp_df)} rows")
print(temp_df.head())

C:\Users\H59045\AppData\Local\Temp\1\ipykernel_29708\262747822.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  temp_df = pd.read_sql(query, conn)


Loaded 12465887 rows
   disclosure_sequence_number  issuer_id state_code   msa  \
0                  1023444403       3871         NJ  None   
1                  1023450796       3774         TN  None   
2                  1023451366       4042         VA  None   
3                  1023503920       3162         TX  None   
4                  1023505445       3162         AR  None   

   original_principal_balance  loan_interest_rate    ltv  credit_score  \
0                    119000.0               8.250  97.51           NaN   
1                     79000.0               8.375  97.62           NaN   
2                     48000.0               8.375  98.95         664.0   
3                     24000.0               8.625  73.85         668.0   
4                     68000.0               8.750  99.78         809.0   

   loan_origination_date down_payment_assistance first_time_home_buyer  \
0                    NaN                     NaN                   NaN   
1                  

In [6]:
static_df = temp_df.copy()

static_df['created_at'] = datetime.now()
static_df['last_updated'] = datetime.now()

print(static_df.shape)

(12465887, 15)


In [7]:
existing_keys_query = "SELECT disclosure_sequence_number FROM gnma_static_table"
existing_keys = pd.read_sql(existing_keys_query, conn)['disclosure_sequence_number'].tolist()

new_loans = static_df[~static_df['disclosure_sequence_number'].isin(existing_keys)]
existing_loans = static_df[static_df['disclosure_sequence_number'].isin(existing_keys)]

print(f"New Count: {len(new_loans)}")
print(f"Existing Count: {len(existing_loans)}")

C:\Users\H59045\AppData\Local\Temp\1\ipykernel_29708\3058999004.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  existing_keys = pd.read_sql(existing_keys_query, conn)['disclosure_sequence_number'].tolist()


New Count: 12465887
Existing Count: 0


In [8]:
if len(new_loans) > 0:
    insert_query ="""INSERT INTO gnma_static_table VALUES %s"""

    new_loans_tuples = [tuple(row) for row in new_loans.values]

    execute_values(cur, insert_query, new_loans_tuples)
    conn.commit()

    print(f"Inserted {len(new_loans)} loans")
else:
    print("No new loans")

UniqueViolation: duplicate key value violates unique constraint "gnma_static_table_pkey"
DETAIL:  Key (disclosure_sequence_number)=(1023444403) already exists.


In [ ]:
if len(existing_loans) > 0:
    for idx, row in existing_loans.iterrows():
        update_query = """
                        UPDATE gnma_static_table
                        SET disclosure_sequence_number = %s,
                        issuer_id = %s,
                        state_code = %s,
                        msa = %s,
                        original_principal_balance = %s,
                        loan_interest_rate = %s,
                        ltv = %s,
                        credit_score = %s,
                        loan_origination_date = %s,
                        down_payment_assistance = %s,
                        first_time_home_buyer = %s,
                        seller_issuer_id,agency = %s,
                        created_at = %s,
                        last_update = %s
                        WHERE disclosure_sequence_number = %s"""

        cur.execute(update_query, tuple(row))

    conn.commit()
    print(f"Updateds {len(existing_loans)} rows")
else:
    print("No loans to update")